## Propósito
Inspeccionar **todos los CSV producidos en la Etapa 1**. Generamos perfiles por columna y un resumen de nulos, folios `SD`, folios repetidos, coincidencias de alcaldía y filas idénticas. **No eliminamos registros**.

## Perfilamiento por archivo

In [ ]:
from pathlib import Path
import pandas as pd
SALIDA=Path("salidas")
FUENTE="SSC"
archivos=sorted(SALIDA.glob(f"{FUENTE}_*_ETAPA1.csv"))
if not archivos:
    print("No hay salidas de Etapa 1. Ejecutar primero el notebook 01 y resolver advertencias.")
resultados=[]
for archivo in archivos:
    df=pd.read_csv(archivo,dtype=str,low_memory=False)
    fol=next((c for c in df.columns if c.lower()=="folio"),None)
    alc=next((c for c in df.columns if c.lower() in ("alcaldia","alcaldia_catalogo","alcaldia_inicio")),None)
    sd=int(df[fol].fillna("").str.strip().str.upper().eq("SD").sum()) if fol else None
    rep=int(df[fol].fillna("").duplicated(keep=False).sum()) if fol else None
    rep_alc=int(df.duplicated(subset=[fol,alc],keep=False).sum()) if fol and alc else None
    resumen={"archivo":archivo.name,"filas":len(df),"columnas":len(df.columns),"nulos":int(df.isna().sum().sum()),"folio_SD":sd,"filas_folio_repetido":rep,"filas_folio_alcaldia_repetidos":rep_alc,"filas_identicas":int(df.duplicated(keep=False).sum())}
    resultados.append(resumen)
    perfil=pd.DataFrame({"tipo":df.dtypes.astype(str),"nulos":df.isna().sum(),"pct_nulos":(df.isna().mean()*100).round(2),"unicos":df.nunique(dropna=True)})
    perfil.to_csv(SALIDA/f"perfil_{archivo.stem}.csv",encoding="utf-8-sig")
reporte=pd.DataFrame(resultados)
display(reporte)
reporte.to_csv(SALIDA/f"{FUENTE}_validacion_etapa2.csv",index=False,encoding="utf-8-sig")
